# Badminton Shot Classification - Video-Based (2D CNN + LSTM)

**Expected Accuracy:** 70-80% (vs 38% with pose-only)

**Architecture:** ResNet18 (pretrained) + BiLSTM

**Dataset:** 18,167 video clips (Clear, Drive, Drop, Lift, Smash)

---

## Instructions

1. **Get your data:**
   - **Option A (Recommended):** Download from GCS `gs://iti123storage/videos/clips/` (see cells below)
   - **Option B:** Upload clips to Google Drive and mount it

2. **Runtime:**
   - Runtime > Change runtime type > GPU (**L4** or T4)

3. **Run all cells:**
   - Runtime > Run all
   - First run will download clips from GCS (~5-10 minutes)

4. **Expected time:**
   - **L4 GPU:** 2-3 hours (after download) - 2x faster than T4
   - **T4 GPU:** 4-6 hours (after download)

---

## 1. Setup

In [ ]:
# Check GPU
!nvidia-smi

In [ ]:
# Keep Colab session alive (prevents idle timeout)
# Run this cell to prevent 90-minute idle disconnection
import IPython
from google.colab import output

display(IPython.display.Javascript('''
 function ClickConnect(){
   btn = document.querySelector("colab-connect-button");
   if (btn != null){
     console.log("Click colab-connect-button"); 
     btn.click();
   }
   btn = document.querySelector('#ok');
   if (btn != null){
     console.log("Click reconnect");
     btn.click();
   }
 }
 
 setInterval(ClickConnect, 60000)
'''))

print("✓ Auto-reconnect enabled (checks every 60 seconds)")

In [ ]:
# Install dependencies
!pip install -q opencv-python-headless tqdm scikit-learn

## 2. Data Download - Choose One Option

### Option A: GCS Download (Recommended)

In [ ]:
# Download clips from Google Cloud Storage
import subprocess
import os

# GCS bucket path
GCS_PATH = "gs://iti123storage/videos/clips/"
LOCAL_PATH = "/content/data/clips"

print(f"Downloading from {GCS_PATH}...")
print(f"This will take ~5-10 minutes for 18,167 clips")
print("="*70)

# Authenticate (uncomment if needed)
# from google.colab import auth
# auth.authenticate_user()

# Create local directory
os.makedirs(LOCAL_PATH, exist_ok=True)

# Download using gsutil
try:
    cmd = f"gsutil -m cp -r {GCS_PATH}* {LOCAL_PATH}/"
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    
    if result.returncode == 0:
        print("✓ Download complete!")
        
        # Verify download
        total_files = 0
        for class_name in ['Clear', 'Drive', 'Drop', 'Lift', 'Smash']:
            class_path = os.path.join(LOCAL_PATH, class_name)
            if os.path.exists(class_path):
                count = len([f for f in os.listdir(class_path) if f.endswith('.mp4')])
                print(f"  {class_name:8s}: {count:5d} clips")
                total_files += count
        
        print(f"\nTotal: {total_files} clips downloaded")
        DATA_ROOT = LOCAL_PATH
        print(f"\n✓ DATA_ROOT set to: {DATA_ROOT}")
    else:
        print(f"❌ Download failed: {result.stderr}")
        print("\nTroubleshooting:")
        print("1. Uncomment auth lines above")
        print("2. Use Option B (Google Drive) below")
except Exception as e:
    print(f"❌ Error: {e}")

### Option B: Google Drive (Alternative)

Skip this if Option A worked above.

In [ ]:
# Uncomment to use Google Drive instead:
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_ROOT = '/content/drive/MyDrive/iti123_data/clips'  # Update path!
# print(f"Using Google Drive: {DATA_ROOT}")

## 3. Configuration

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import torchvision.transforms as transforms
import torchvision.models as models

import cv2
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from collections import defaultdict
import json

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Configuration
# DATA_ROOT should be set by Option A or B above
if 'DATA_ROOT' not in locals():
    DATA_ROOT = '/content/data/clips'  # Default from GCS

CONFIG = {
    'data_root': DATA_ROOT,
    'num_frames': 16,
    'frame_size': (224, 224),
    'num_classes': 5,
    'class_names': ['Clear', 'Drive', 'Drop', 'Lift', 'Smash'],
    
    # L4 GPU optimized settings
    'batch_size': 64,              # L4 has 24GB VRAM (2x T4)
    'num_epochs': 50,
    'learning_rate': 0.0001,
    'weight_decay': 0.0001,
    'early_stopping_patience': 10,
    
    'lstm_hidden_size': 256,
    'lstm_num_layers': 2,
    'lstm_dropout': 0.5,
    'freeze_cnn': True,
    
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'num_workers': 4,              # L4 can handle more workers
    
    'use_focal_loss': True,
    'focal_gamma': 2.0,
    
    'output_dir': '/content/models',
    'save_best_model': True,
}

os.makedirs(CONFIG['output_dir'], exist_ok=True)

print("Configuration:")
print(f"  Data root: {CONFIG['data_root']}")
print(f"  Device: {CONFIG['device']}")
print(f"  Batch size: {CONFIG['batch_size']} (L4 optimized)")
print(f"  Num workers: {CONFIG['num_workers']}")

## 4. Dataset Class

In [ ]:
class BadmintonVideoDataset(Dataset):
    def __init__(self, video_paths, labels, num_frames=16, frame_size=(224, 224), augment=False):
        self.video_paths = video_paths
        self.labels = labels
        self.num_frames = num_frames
        self.frame_size = frame_size
        
        normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        
        if augment:
            self.transform = transforms.Compose([
                transforms.ToPILImage(),
                transforms.RandomHorizontalFlip(p=0.5),
                transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
                transforms.RandomRotation(degrees=5),
                transforms.ToTensor(),
                normalize
            ])
        else:
            self.transform = transforms.Compose([
                transforms.ToPILImage(),
                transforms.ToTensor(),
                normalize
            ])
    
    def __len__(self):
        return len(self.video_paths)
    
    def load_video_frames(self, video_path):
        cap = cv2.VideoCapture(str(video_path))
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        
        if total_frames == 0:
            cap.release()
            return torch.zeros(self.num_frames, 3, *self.frame_size)
        
        indices = np.linspace(0, total_frames - 1, self.num_frames, dtype=int)
        frames = []
        
        for idx in indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
            ret, frame = cap.read()
            
            if not ret:
                if len(frames) > 0:
                    frames.append(frames[-1])
                else:
                    frames.append(torch.zeros(3, *self.frame_size))
                continue
            
            frame = cv2.resize(frame, self.frame_size)
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frame_tensor = self.transform(frame)
            frames.append(frame_tensor)
        
        cap.release()
        return torch.stack(frames)
    
    def __getitem__(self, idx):
        return self.load_video_frames(self.video_paths[idx]), self.labels[idx]

print("✓ Dataset class defined")

## 5. Model Architecture

In [ ]:
class CNN_LSTM_Classifier(nn.Module):
    def __init__(self, num_classes=5, lstm_hidden_size=256, lstm_num_layers=2,
                 lstm_dropout=0.5, freeze_cnn=True):
        super().__init__()
        
        resnet = models.resnet18(pretrained=True)
        self.cnn = nn.Sequential(*list(resnet.children())[:-1])
        
        if freeze_cnn:
            for param in self.cnn.parameters():
                param.requires_grad = False
        
        self.cnn_feature_dim = 512
        
        self.lstm = nn.LSTM(
            input_size=self.cnn_feature_dim,
            hidden_size=lstm_hidden_size,
            num_layers=lstm_num_layers,
            batch_first=True,
            dropout=lstm_dropout if lstm_num_layers > 1 else 0,
            bidirectional=True
        )
        
        lstm_output_dim = lstm_hidden_size * 2
        self.classifier = nn.Sequential(
            nn.Dropout(lstm_dropout),
            nn.Linear(lstm_output_dim, 128),
            nn.ReLU(),
            nn.Dropout(lstm_dropout),
            nn.Linear(128, num_classes)
        )
    
    def forward(self, x):
        batch_size, num_frames, C, H, W = x.shape
        x = x.view(batch_size * num_frames, C, H, W)
        cnn_features = self.cnn(x).view(batch_size * num_frames, -1)
        cnn_features = cnn_features.view(batch_size, num_frames, -1)
        lstm_out, _ = self.lstm(cnn_features)
        return self.classifier(lstm_out[:, -1, :])

print("✓ Model class defined")

## 6. Focal Loss

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction
    
    def forward(self, inputs, targets):
        ce_loss = nn.functional.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = (1 - pt) ** self.gamma * ce_loss
        
        if self.alpha is not None:
            focal_loss = self.alpha[targets] * focal_loss
        
        return focal_loss.mean() if self.reduction == 'mean' else focal_loss.sum()

print("✓ Focal Loss defined")

## 7. Load Dataset

In [ ]:
def load_dataset(data_root, class_names):
    data_root = Path(data_root)
    video_paths = []
    labels = []
    class_counts = defaultdict(int)
    
    for class_idx, class_name in enumerate(class_names):
        class_dir = data_root / class_name
        if not class_dir.exists():
            print(f"Warning: {class_dir} does not exist!")
            continue
        
        mp4_files = list(class_dir.glob("*.mp4"))
        video_paths.extend(mp4_files)
        labels.extend([class_idx] * len(mp4_files))
        class_counts[class_name] = len(mp4_files)
    
    return video_paths, labels, class_counts

def compute_class_weights(labels, num_classes):
    counts = np.bincount(labels, minlength=num_classes)
    weights = np.sqrt(len(labels) / counts)
    weights = weights * num_classes / weights.sum()
    return torch.FloatTensor(weights)

print("Loading dataset...")
video_paths, labels, class_counts = load_dataset(CONFIG['data_root'], CONFIG['class_names'])

print(f"\nTotal videos: {len(video_paths)}")
print("\nClass distribution:")
for class_name in CONFIG['class_names']:
    count = class_counts[class_name]
    print(f"  {class_name:8s}: {count:5d} ({100*count/len(labels):.1f}%)")

## 7B. Frame Preprocessing (Optional - For Fast Training)

**Skip this entire section if you want to start training immediately.**

**Use this section if:**
- Training is very slow (>20s per batch)
- Video decoding is the bottleneck
- You want 100x faster training

**What this does:**
- Pre-extracts 16 frames from each video
- Saves as .npy files (~4.7 GB for 18K videos)
- Takes 20-30 minutes with parallel processing
- Makes training 100x faster (59s → 0.5s per batch)

**How to use:** Uncomment and run the cells below in order.

In [ ]:
# Frame Extraction Functions (with parallel processing)

import os
import cv2
import numpy as np
from pathlib import Path
from tqdm.notebook import tqdm
from multiprocessing import Pool, cpu_count

def extract_frames_from_video(video_path, num_frames=16, frame_size=(224, 224)):
    """Extract and resize frames from a single video."""
    cap = cv2.VideoCapture(str(video_path))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    if total_frames == 0:
        cap.release()
        return None
    
    indices = np.linspace(0, total_frames - 1, num_frames, dtype=int)
    frames = []
    
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        
        if not ret:
            if len(frames) > 0:
                frames.append(frames[-1])
            else:
                frames.append(np.zeros((frame_size[1], frame_size[0], 3), dtype=np.uint8))
            continue
        
        frame = cv2.resize(frame, frame_size)
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frames.append(frame)
    
    cap.release()
    
    if len(frames) == 0:
        return None
    
    return np.array(frames, dtype=np.uint8)


def process_single_video(args):
    """Process a single video (for parallel execution)."""
    video_path, output_dir, num_frames, frame_size = args
    
    video_id = video_path.stem
    class_name = video_path.parent.name
    npy_filename = f"{class_name}_{video_id}.npy"
    npy_path = Path(output_dir) / npy_filename
    
    # Skip if already exists
    if npy_path.exists():
        return npy_path, None
    
    # Extract frames
    frames = extract_frames_from_video(video_path, num_frames, frame_size)
    
    if frames is None:
        return None, str(video_path)
    
    # Save
    np.save(npy_path, frames)
    return npy_path, None


def preprocess_all_videos_parallel(video_paths, output_dir, num_frames=16, 
                                   frame_size=(224, 224), num_workers=None):
    """
    Pre-extract frames from all videos in parallel.
    
    Args:
        video_paths: List of video file paths
        output_dir: Directory to save .npy files
        num_frames: Number of frames per video
        frame_size: Target frame size (W, H)
        num_workers: Number of parallel workers (default: CPU count - 1)
    
    Returns:
        npy_paths: List of .npy file paths
        failed_videos: List of failed video paths
    """
    os.makedirs(output_dir, exist_ok=True)
    
    if num_workers is None:
        num_workers = max(1, cpu_count() - 1)
    
    print(f"Extracting frames from {len(video_paths)} videos...")
    print(f"Output directory: {output_dir}")
    print(f"Parallel workers: {num_workers}")
    print(f"Estimated time: 20-30 minutes for 18K videos")
    print("="*70)
    
    # Prepare arguments
    args_list = [(vp, output_dir, num_frames, frame_size) for vp in video_paths]
    
    # Process in parallel with progress bar
    npy_paths = []
    failed_videos = []
    
    with Pool(num_workers) as pool:
        results = list(tqdm(
            pool.imap(process_single_video, args_list),
            total=len(video_paths),
            desc='Extracting'
        ))
    
    # Collect results
    for npy_path, failed_path in results:
        if npy_path:
            npy_paths.append(npy_path)
        if failed_path:
            failed_videos.append(failed_path)
    
    print("\n" + "="*70)
    print(f"✓ Extracted {len(npy_paths)} videos")
    if failed_videos:
        print(f"⚠️  Failed: {len(failed_videos)} videos")
        print(f"First few failures: {failed_videos[:5]}")
    
    # Calculate storage size
    if npy_paths:
        sample_size = os.path.getsize(npy_paths[0]) / 1024 / 1024  # MB
        total_size = sample_size * len(npy_paths)
        print(f"\nStorage used: {total_size:.1f} MB (~{total_size/1024:.1f} GB)")
    
    return npy_paths, failed_videos

print("✓ Parallel frame extraction functions loaded")

In [ ]:
# Run Frame Extraction (UNCOMMENT TO USE)
# This will take 20-30 minutes with parallel processing

# FRAMES_DIR = '/content/data/frames'
# 
# print("Starting parallel frame extraction...")
# npy_paths, failed = preprocess_all_videos_parallel(
#     video_paths,
#     FRAMES_DIR,
#     num_frames=CONFIG['num_frames'],
#     frame_size=CONFIG['frame_size'],
#     num_workers=None  # Uses CPU count - 1
# )
# 
# print(f"\n✓ Preprocessing complete!")
# print(f"✓ Frames saved to: {FRAMES_DIR}")
# print(f"✓ Ready for fast training!")

print("Frame extraction ready.")
print("Uncomment above to run parallel extraction (~20-30 min).")

In [ ]:
# Fast Dataset Class for Pre-extracted Frames

class BadmintonFramesDataset(Dataset):
    """
    Dataset for pre-extracted frames (much faster than video decoding).
    """
    def __init__(self, npy_paths, labels, augment=False):
        self.npy_paths = npy_paths
        self.labels = labels
        self.augment = augment
        
        normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                                        std=[0.229, 0.224, 0.225])
        
        if augment:
            self.transform = transforms.Compose([
                transforms.ToPILImage(),
                transforms.RandomHorizontalFlip(p=0.5),
                transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
                transforms.RandomRotation(degrees=5),
                transforms.ToTensor(),
                normalize
            ])
        else:
            self.transform = transforms.Compose([
                transforms.ToPILImage(),
                transforms.ToTensor(),
                normalize
            ])
    
    def __len__(self):
        return len(self.npy_paths)
    
    def __getitem__(self, idx):
        frames = np.load(self.npy_paths[idx])  # (T, H, W, C)
        label = self.labels[idx]
        
        frames_tensor = []
        for frame in frames:
            frame_tensor = self.transform(frame)
            frames_tensor.append(frame_tensor)
        
        return torch.stack(frames_tensor), label

print("✓ Fast frames dataset class loaded")

In [ ]:
# Create Fast Data Loaders (UNCOMMENT AFTER EXTRACTION)
# Run this AFTER frame extraction completes above

# # 1. Create train/val/test splits with .npy paths
# from sklearn.model_selection import train_test_split
# 
# train_npy_paths, temp_npy_paths, train_labels, temp_labels = train_test_split(
#     npy_paths, labels, test_size=0.3, random_state=42, stratify=labels
# )
# 
# val_npy_paths, test_npy_paths, val_labels, test_labels = train_test_split(
#     temp_npy_paths, temp_labels, test_size=0.5, random_state=42, stratify=temp_labels
# )
# 
# print(f"Train: {len(train_npy_paths)} samples")
# print(f"Val:   {len(val_npy_paths)} samples")
# print(f"Test:  {len(test_npy_paths)} samples")
# 
# # 2. Create fast datasets
# train_dataset = BadmintonFramesDataset(train_npy_paths, train_labels, augment=True)
# val_dataset = BadmintonFramesDataset(val_npy_paths, val_labels, augment=False)
# test_dataset = BadmintonFramesDataset(test_npy_paths, test_labels, augment=False)
# 
# # 3. Create data loaders
# train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'],
#                          shuffle=True, num_workers=CONFIG['num_workers'], pin_memory=True)
# 
# val_loader = DataLoader(val_dataset, batch_size=CONFIG['batch_size'],
#                        shuffle=False, num_workers=CONFIG['num_workers'], pin_memory=True)
# 
# test_loader = DataLoader(test_dataset, batch_size=CONFIG['batch_size'],
#                         shuffle=False, num_workers=CONFIG['num_workers'], pin_memory=True)
# 
# # 4. Compute class weights
# class_weights = compute_class_weights(train_labels, CONFIG['num_classes'])
# 
# print(f"\n✓ Fast data loaders created!")
# print(f"  Train batches: {len(train_loader)}")
# print(f"  Val batches:   {len(val_loader)}")
# print(f"  Test batches:  {len(test_loader)}")
# print(f"\n✓ Now SKIP to Section 10 (Model Training)")

print("Fast data loaders ready.")
print("Uncomment above after frame extraction completes.")

### Section 7B Summary

**If you used frame preprocessing above:**
1. ✓ Frames extracted to .npy files
2. ✓ Fast data loaders created
3. **SKIP Section 8 and 9** (they use slow video loading)
4. **GO TO Section 10** (Model Training)

**If you skipped preprocessing:**
1. Continue to Section 8 (Train/Val/Test Split)
2. Section 9 will create standard video loaders (slower but works)

**Tip:** Use `Ctrl + /` or `Cmd + /` to uncomment code blocks quickly.

## 8. Train/Val/Test Split

In [ ]:
train_paths, temp_paths, train_labels, temp_labels = train_test_split(
    video_paths, labels, test_size=0.3, random_state=42, stratify=labels
)

val_paths, test_paths, val_labels, test_labels = train_test_split(
    temp_paths, temp_labels, test_size=0.5, random_state=42, stratify=temp_labels
)

print(f"Train: {len(train_paths)} videos")
print(f"Val:   {len(val_paths)} videos")
print(f"Test:  {len(test_paths)} videos")

class_weights = compute_class_weights(train_labels, CONFIG['num_classes'])
print(f"\nClass weights (softened):")
for i, name in enumerate(CONFIG['class_names']):
    print(f"  {name:8s}: {class_weights[i]:.3f}")

## 9. Create Data Loaders

In [ ]:
train_dataset = BadmintonVideoDataset(train_paths, train_labels, 
                                      num_frames=CONFIG['num_frames'],
                                      frame_size=CONFIG['frame_size'],
                                      augment=True)

val_dataset = BadmintonVideoDataset(val_paths, val_labels,
                                    num_frames=CONFIG['num_frames'],
                                    frame_size=CONFIG['frame_size'],
                                    augment=False)

test_dataset = BadmintonVideoDataset(test_paths, test_labels,
                                     num_frames=CONFIG['num_frames'],
                                     frame_size=CONFIG['frame_size'],
                                     augment=False)

train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'],
                         shuffle=True, num_workers=CONFIG['num_workers'], pin_memory=True)

val_loader = DataLoader(val_dataset, batch_size=CONFIG['batch_size'],
                       shuffle=False, num_workers=CONFIG['num_workers'], pin_memory=True)

test_loader = DataLoader(test_dataset, batch_size=CONFIG['batch_size'],
                        shuffle=False, num_workers=CONFIG['num_workers'], pin_memory=True)

print(f"✓ Data loaders created")
print(f"  Train batches: {len(train_loader)}")
print(f"  Val batches:   {len(val_loader)}")
print(f"  Test batches:  {len(test_loader)}")

## 10. Create Model & Training Setup

In [ ]:
model = CNN_LSTM_Classifier(
    num_classes=CONFIG['num_classes'],
    lstm_hidden_size=CONFIG['lstm_hidden_size'],
    lstm_num_layers=CONFIG['lstm_num_layers'],
    lstm_dropout=CONFIG['lstm_dropout'],
    freeze_cnn=CONFIG['freeze_cnn']
).to(CONFIG['device'])

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

criterion = FocalLoss(alpha=class_weights.to(CONFIG['device']), gamma=CONFIG['focal_gamma'])
optimizer = optim.Adam(model.parameters(), lr=CONFIG['learning_rate'], 
                      weight_decay=CONFIG['weight_decay'])
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CONFIG['num_epochs'])

print("✓ Training setup complete")

In [ ]:
# === PERFORMANCE CHECK ===
# Run this to verify GPU is working and estimate training time
import time

print("="*70)
print("PERFORMANCE DEBUG")
print("="*70)

# 1. Check GPU
print(f"\n1. GPU Status:")
print(f"   Device: {CONFIG['device']}")
print(f"   CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   GPU name: {torch.cuda.get_device_name(0)}")
    print(f"   GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# 2. Check data location
print(f"\n2. Data Location:")
print(f"   Data root: {CONFIG['data_root']}")
is_drive = '/drive/' in CONFIG['data_root']
print(f"   Using Google Drive: {is_drive} {'❌ SLOW!' if is_drive else '✓ Good'}")

# 3. Check data loader
print(f"\n3. Data Loader:")
print(f"   Batch size: {CONFIG['batch_size']}")
print(f"   Num workers: {CONFIG['num_workers']}")
print(f"   Batches/epoch: {len(train_loader)}")

# 4. Test single batch speed
print(f"\n4. Testing batch loading...")
start = time.time()
batch = next(iter(train_loader))
frames, labels = batch
batch_load_time = time.time() - start
print(f"   Load time: {batch_load_time:.2f}s")
print(f"   Shape: {frames.shape}")

# 5. Test model forward pass
print(f"\n5. Testing GPU computation...")
model_on_gpu = next(model.parameters()).is_cuda
print(f"   Model on GPU: {model_on_gpu}")
frames = frames.to(CONFIG['device'])
start = time.time()
with torch.no_grad():
    output = model(frames)
forward_time = time.time() - start
print(f"   Forward pass: {forward_time:.3f}s")

# 6. Estimate epoch time
batch_time = forward_time * 2.5  # Forward + backward + optimizer
epoch_estimate = (batch_time * len(train_loader)) / 60
total_estimate = epoch_estimate * CONFIG['num_epochs'] / 60

print(f"\n6. Time Estimates:")
print(f"   Per batch: {batch_time:.2f}s")
print(f"   Per epoch: {epoch_estimate:.1f} minutes")
print(f"   Total (50 epochs): {total_estimate:.1f} hours")

print("\n" + "="*70)
print("DIAGNOSIS:")
print("="*70)

issues = []
if not torch.cuda.is_available():
    issues.append("❌ GPU NOT AVAILABLE - Runtime not set to GPU!")
if not model_on_gpu:
    issues.append("❌ MODEL NOT ON GPU - Check device setting!")
if is_drive:
    issues.append("❌ READING FROM GOOGLE DRIVE - Will be VERY slow! Use GCS download.")
if batch_load_time > 5:
    issues.append("⚠️  Batch loading slow - Check data location")
if forward_time > 1.0:
    issues.append("⚠️  Forward pass slow - GPU may not be utilized")
if epoch_estimate > 10:
    issues.append("❌ TRAINING WILL TAKE TOO LONG - Fix issues above!")

if issues:
    print("\n⚠️  ISSUES FOUND:\n")
    for issue in issues:
        print(f"   {issue}")
    print("\n   Fix these before training!")
else:
    print("✓ All checks passed!")
    print(f"✓ Training will take ~{total_estimate:.1f} hours")
    if total_estimate < 4:
        print("✓ Speed looks good! Ready to train.")

print("="*70)

### Performance Check

**IMPORTANT:** Run this cell to verify GPU is working and training will be fast!

## 11. Training Functions

In [ ]:
def train_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for frames, labels in tqdm(dataloader, desc='Training'):
        frames, labels = frames.to(device), labels.to(device)
        
        optimizer.zero_grad()
        logits = model(frames)
        loss = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        total_loss += loss.item()
        _, predicted = logits.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    
    return total_loss / len(dataloader), 100. * correct / total

def validate_epoch(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for frames, labels in tqdm(dataloader, desc='Validation'):
            frames, labels = frames.to(device), labels.to(device)
            logits = model(frames)
            loss = criterion(logits, labels)
            
            total_loss += loss.item()
            _, predicted = logits.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    f1 = f1_score(all_labels, all_preds, average='macro')
    return total_loss / len(dataloader), 100. * correct / total, f1, all_preds, all_labels

print("✓ Training functions defined")

## 12. Train Model

In [ ]:
best_val_f1 = 0
patience_counter = 0
history = defaultdict(list)

# Checkpoint settings
checkpoint_dir = CONFIG['output_dir']
checkpoint_path = os.path.join(checkpoint_dir, 'checkpoint.pth')

# Try to resume from checkpoint if exists
start_epoch = 0
if os.path.exists(checkpoint_path):
    print("Found checkpoint! Resuming training...")
    checkpoint = torch.load(checkpoint_path)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    start_epoch = checkpoint['epoch'] + 1
    best_val_f1 = checkpoint['best_val_f1']
    history = checkpoint['history']
    patience_counter = checkpoint['patience_counter']
    print(f"✓ Resumed from epoch {start_epoch}")

print("="*70)
print("Starting training...")
print("="*70)

for epoch in range(start_epoch, CONFIG['num_epochs']):
    print(f"\nEpoch {epoch+1}/{CONFIG['num_epochs']}")
    
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, CONFIG['device'])
    val_loss, val_acc, val_f1, val_preds, val_labels_list = validate_epoch(model, val_loader, criterion, CONFIG['device'])
    scheduler.step()
    
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['val_f1'].append(val_f1)
    
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
    print(f"Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.2f}% | Val F1: {val_f1:.4f}")
    print(f"Gap:        {train_acc - val_acc:.2f}%")
    
    # Save best model
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        patience_counter = 0
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'val_f1': val_f1,
            'val_acc': val_acc,
        }, os.path.join(CONFIG['output_dir'], 'best_model.pth'))
        print(f"✓ Saved best model (F1: {val_f1:.4f})")
    else:
        patience_counter += 1
    
    # Save checkpoint every 5 epochs (for crash recovery)
    if (epoch + 1) % 5 == 0:
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'best_val_f1': best_val_f1,
            'history': history,
            'patience_counter': patience_counter,
        }, checkpoint_path)
        print(f"✓ Checkpoint saved (epoch {epoch+1})")
    
    # Early stopping
    if patience_counter >= CONFIG['early_stopping_patience']:
        print(f"\nEarly stopping at epoch {epoch+1}")
        break

# Save final checkpoint
torch.save({
    'epoch': epoch,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'scheduler_state_dict': scheduler.state_dict(),
    'best_val_f1': best_val_f1,
    'history': history,
    'patience_counter': patience_counter,
}, checkpoint_path)

print("\nTraining complete!")
print(f"Best validation F1: {best_val_f1:.4f}")

## 13. Plot Training Curves

In [ ]:
pd.DataFrame(history).to_csv(os.path.join(CONFIG['output_dir'], 'training_history.csv'), index=False)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(history['train_loss'], label='Train')
axes[0].plot(history['val_loss'], label='Val')
axes[0].set_title('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history['train_acc'], label='Train')
axes[1].plot(history['val_acc'], label='Val')
axes[1].set_title('Accuracy (%)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].plot(history['val_f1'])
axes[2].set_title('Validation F1')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(CONFIG['output_dir'], 'training_curves.png'), dpi=150)
plt.show()

print(f"Best validation F1: {best_val_f1:.4f}")

## 14. Evaluate on Test Set

In [ ]:
checkpoint = torch.load(os.path.join(CONFIG['output_dir'], 'best_model.pth'))
model.load_state_dict(checkpoint['model_state_dict'])

test_loss, test_acc, test_f1, test_preds, test_labels_list = validate_epoch(
    model, test_loader, criterion, CONFIG['device']
)

print("\n" + "="*70)
print("Test Set Results")
print("="*70)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Acc:  {test_acc:.2f}%")
print(f"Test F1:   {test_f1:.4f}")

print("\n" + "="*70)
print("Classification Report:")
print("="*70)
print(classification_report(test_labels_list, test_preds,
                          target_names=CONFIG['class_names'], digits=3))

cm = confusion_matrix(test_labels_list, test_preds)

plt.figure(figsize=(8, 6))
plt.imshow(cm, interpolation='nearest', cmap='Blues')
plt.title('Confusion Matrix')
plt.colorbar()
tick_marks = np.arange(len(CONFIG['class_names']))
plt.xticks(tick_marks, CONFIG['class_names'], rotation=45)
plt.yticks(tick_marks, CONFIG['class_names'])

thresh = cm.max() / 2.
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, format(cm[i, j], 'd'),
                ha="center", va="center",
                color="white" if cm[i, j] > thresh else "black")

plt.ylabel('True label')
plt.xlabel('Predicted label')
plt.tight_layout()
plt.savefig(os.path.join(CONFIG['output_dir'], 'confusion_matrix.png'), dpi=150)
plt.show()

## 15. Download Results

In [ ]:
from google.colab import files

print("Downloading files...")
files.download(os.path.join(CONFIG['output_dir'], 'best_model.pth'))
files.download(os.path.join(CONFIG['output_dir'], 'training_history.csv'))
files.download(os.path.join(CONFIG['output_dir'], 'training_curves.png'))
files.download(os.path.join(CONFIG['output_dir'], 'confusion_matrix.png'))

print("✓ Download complete!")

---

## Summary

**Expected Results:**
- Test Accuracy: 70-80%
- Test F1: 0.65-0.75
- Much better than pose-only (38%)

**Files Downloaded:**
1. `best_model.pth` - Trained model
2. `training_history.csv` - Metrics per epoch
3. `training_curves.png` - Training plots
4. `confusion_matrix.png` - Confusion matrix

**Next Steps:**
- Use `predict_video.py` locally for inference
- If accuracy >70%: Deploy or fine-tune CNN
- If accuracy <65%: Try unfreezing CNN or X3D model

---